# YOLOv1
1. two-stage 和 one-stage 方法
    + two-stage：先对框进行预选，然后输出结果
<img src="./images/YOLO/two-stage.png">
    + one-stage：将问题建模为回归任务，即拟合$(x,y,w,h)$
<img src="./images/YOLO/one-stage.png">
2. 指标
    + mAP：平均准确率，指标是指每个类别的平均精度
    + IoU：交并比，衡量预测框和真实框的重合程度
3. 核心思想
<img src="./images/YOLO/v1_idea.png">
4. 网络架构
    + $7\times7\times30$：$S=7$，$30=5+5+20$
    + $5=4+1$，$4$ 为框，$1$ 为置信度，预测两个框
    + $20$ 为类别
<img src="./images/YOLO/v1_net.png">
5. 损失函数
<img src="./images/YOLO/v1_loss.png">
6. NMS：非极大值抑制，去除重复框
7. 优点：快速，简单
8. 问题
    + 每个 cell 只能预测一个类别，无法解决重叠的情况
    + 小物体检测效果一般，长宽比可选但单一

# YOLOv2 改进
1. 舍弃全连接层以及Dropout，卷积后加入 Batch Normalization
2. 更大的分辨率：V1 训练用的 $224\times224$，测试用的 $448\times448$，V2 训练时额外对 $448\times448$ 进行了 10 次微调
3. 网络结构：DarkNet
    + 没有 FC 层，只有卷积层
    + $1\times1$ 卷积节省参数

<img src="./images/YOLO/v2_net.png">

4. kmeans 聚类提取先验框：$d(box,centroid)=1-IoU(box,centroid)$
5. 引入先验框，使得预测的 box 数量更多 $(13\times13\times n)$
6. bbox 的计算方式可能导致收敛问题，模型不稳定，尤其是刚开始训练的时候
    + $x=x_p+w_p\cdot tx$
    + $y=y_p+h_p\cdot ty$
    + 因此将偏移量更改为相对 grid cell 的偏移量
    + $b_x=\sigma(t_x)+c_x$, $b_y=\sigma(t_y)+c_y$, $b_w=p_w\cdot \exp(t_w)$, $b_h=p_h\cdot \exp(t_h)$

<img src="./images/YOLO/v2_bbox.png">

7. 特征融合：由于最后一层的感受野太大，可能丢失小目标，需要融合之前的特征
8. 多尺度：由于都是卷积层，可以多尺度训练（一定 iterations 后改变输入图像大小），提高检测能力

# YOLOv3
1. 多 scale
    + 不同尺度的特征图预测不同尺度的 box
<img src="./images/YOLO/v3_scale.png">
    + 图像金字塔

<img src="./images/YOLO/v3_pyramid.png">

2. 残差连接
3. 核心网络架构：没有池化和全连接
<img src="./images/YOLO/v3_net.png">
4. 先验框设计
    + $13\times13$：$(116, 90), (156, 198), (373, 326)$
    + $26\times26$：$(30, 61), (62, 45), (59, 119)$
    + $52\times52$：$(10, 13), (16, 30), (33, 23)$
5. softmax 层替代：一个物体可能有多个标签
    + 使用 logistic 激活函数，输出为 $(0,1)$，表示每个类别的概率